# Step-sampling follow-up: strengthen (or kill) the late-sampling win

`sampling.ipynb` found **S3_late +0.0051, t 2.12** over the log baseline — borderline, and
with two open questions before it justifies a 6.5h cloud run:

1. **Is it placement or just +20% rows?** S3_late trains on ~315k rows/fold vs log's ~263k.
   S1_uniform (same ~315k) gained nothing, which already points at placement — here we
   nail it two ways: (a) paired S3_late vs S1_uniform (equal budget), and (b) `S_lean`, a
   late scheme down-sampled to log's row budget, paired vs log.
2. **Is the driver "more/cleaner positives"?** If so, pushing sampling *later* (higher
   pos-rate) should keep helping until it plateaus. Sweep sharper-late schemes and read the
   pos-rate -> gain curve.

Same 5 folds, same pointwise CatBoost, `paired_compare`, believe at t >= 2.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostClassifier

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

d = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = d['X'], d['y'], d['sid'], d['time']
log_mask = d['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
folds = CV.series_folds(sid, n_splits=5, seed=0)
uids, starts = np.unique(sid, return_index=True)
bounds = np.append(starts, len(sid)); lengths = np.diff(bounds)
Lrow = pd.Series(lengths, index=uids).reindex(sid).to_numpy()

# --- reconstruct the prior CVResults so new schemes share the paired baseline ---
from cv_tools import CVResult
prev = json.load(open('sampling_results.json'))
def as_result(rec):
    return CVResult(np.array(rec['fold_scores']), None, rec['oof'], 5)
res = {k: as_result(v) for k, v in prev.items()}
print('loaded prior:', list(res))
print('\nS3_late vs S1_uniform (equal ~315k budget -> isolates PLACEMENT):')
print(CV.paired_compare(res['S3_late'], res['S1_uniform'], 'S3_late', 'S1_uniform'))

loaded prior: ['S0_log', 'S1_uniform', 'S2_triangular', 'S3_late']

S3_late vs S1_uniform (equal ~315k budget -> isolates PLACEMENT):
S3_late - S1_uniform: +0.0040 +/- 0.0059 (sem 0.0026, t +1.52, wins 4/5) -> within noise
  per-fold deltas: +0.0004  -0.0008  +0.0136  +0.0058  +0.0011


In [2]:
def sample_steps(L, m, kind):
    if L <= m:
        return np.arange(L)
    u = np.linspace(0, 1, 512)
    if kind == 'late2':      dens = u ** 2          # sharper late
    elif kind == 'late3':    dens = u ** 3          # sharper still
    elif kind == 'secondhalf':
        idx = np.round(np.linspace(L // 2, L - 1, m)).astype(int)   # uniform over [L/2, L)
        return np.unique(idx)
    else:
        raise ValueError(kind)
    cdf = np.cumsum(dens); cdf /= cdf[-1]
    probs = (np.arange(m) + 0.5) / m
    return np.unique(np.round(np.interp(probs, cdf, u) * (L - 1)).astype(int))


def build_mask(kind, m=40):
    mask = np.zeros(len(sid), bool)
    for k in range(len(uids)):
        mask[bounds[k] + sample_steps(lengths[k], m, kind)] = True
    return mask


# S_lean: the winning 'late' shape (density ∝ step) but down-sampled to log's row budget,
# so the S_lean-vs-log comparison has NO row-count advantage.
def sample_linear_late(L, m):
    if L <= m:
        return np.arange(L)
    u = np.linspace(0, 1, 512); cdf = np.cumsum(u); cdf /= cdf[-1]
    probs = (np.arange(m) + 0.5) / m
    return np.unique(np.round(np.interp(probs, cdf, u) * (L - 1)).astype(int))

# tune m so total rows ~ log's 328,996
target = int(log_mask.sum())
for mm in range(40, 20, -1):
    tot = sum(len(sample_linear_late(L, mm)) for L in lengths)
    if tot <= target:
        lean_m = mm; break
lean_mask = np.zeros(len(sid), bool)
for k in range(len(uids)):
    lean_mask[bounds[k] + sample_linear_late(lengths[k], lean_m)] = True

NEW = {'S4_late2': build_mask('late2'),
       'S5_late3': build_mask('late3'),
       'S6_secondhalf': build_mask('secondhalf'),
       'S_lean_late': lean_mask}
for n, m in NEW.items():
    frac = step[m] / np.maximum(Lrow[m] - 1, 1)
    print(f'{n:16s} rows {m.sum():>8,} | pos-rate {y[m].mean():.3f} | mean frac {frac.mean():.3f}')
print(f'(S_lean uses m={lean_m} to match log budget {target:,})')

S4_late2         rows  391,392 | pos-rate 0.374 | mean frac 0.743
S5_late3         rows  388,435 | pos-rate 0.397 | mean frac 0.792
S6_secondhalf    rows  391,285 | pos-rate 0.375 | mean frac 0.745
S_lean_late      rows  326,092 | pos-rate 0.335 | mean frac 0.663
(S_lean uses m=33 to match log budget 328,996)


In [3]:
CATP = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
            random_seed=0, verbose=0, thread_count=-1)
def cat_fp(train, val):
    m = CatBoostClassifier(**CATP)
    m.fit(train.X, train.y, sample_weight=train.w)
    return m.predict_proba(val.X)[:, 1]

OUT = 'sampling_followup_results.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
for name, mask in NEW.items():
    if name in done:
        res[name] = CVResult(np.array(done[name]['fold_scores']), None, done[name]['oof'], 5)
        print(f'{name}: cached'); continue
    t = time.time()
    print(f'\n=== {name} ===', flush=True)
    r = CV.run_cv(X, y, sid, index, cat_fp, folds=folds, train_row_mask=mask, row_step=step)
    res[name] = r
    done[name] = dict(fold_scores=r.fold_scores.tolist(), mean=r.mean, std=r.std,
                      sem=r.sem, oof=r.oof_score)
    json.dump(done, open(OUT, 'w'), indent=2)
    print(f'{r.summary(name)} | {time.time() - t:.0f}s', flush=True)


=== S4_late2 ===


  fold 0: train   313,426 rows / 8,000 series | val 1,012,996 rows / 2,000 series | TS-AUC 0.5954


  fold 1: train   313,169 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5944


  fold 2: train   312,941 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5912


  fold 3: train   313,099 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6064


...

  fold 1: train   260,941 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5993


  fold 2: train   260,786 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5870


  fold 3: train   260,867 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6051


  fold 4: train   260,734 rows / 8,000 series | val 1,007,487 rows / 2,000 series | TS-AUC 0.6205


S_lean_late        mean 0.6016 +/- 0.0124 (sem 0.0056) | OOF 0.6011 | folds: 0.5963  0.5993  0.5870  0.6051  0.6205 | 255s


In [4]:
print(CV.summarize(res))

prates = {'S0_log': y[log_mask].mean(), 'S1_uniform': 0.254, 'S3_late': 0.334}
for n, m in NEW.items():
    prates[n] = y[m].mean()

print('\n--- pos-rate -> gain vs log (mechanism check) ---')
base = res['S0_log'].mean
for n in ['S0_log', 'S1_uniform', 'S3_late', 'S4_late2', 'S5_late3', 'S6_secondhalf', 'S_lean_late']:
    if n in res:
        pr = prates.get(n, float('nan'))
        print(f'{n:16s} pos-rate {pr:.3f}  mean {res[n].mean:.4f}  delta {res[n].mean - base:+.4f}')

print('\n--- confound control: late shape at LOG budget ---')
print(CV.paired_compare(res['S_lean_late'], res['S0_log'], 'S_lean_late', 'S0_log'))
print('\n--- placement isolated: late vs uniform at equal budget ---')
print(CV.paired_compare(res['S3_late'], res['S1_uniform'], 'S3_late', 'S1_uniform'))

print('\n--- sharper-late vs log baseline ---')
for n in ['S4_late2', 'S5_late3', 'S6_secondhalf']:
    print(CV.paired_compare(res[n], res['S0_log'], n, 'S0_log'))


S3_late            mean 0.6028 +/- 0.0124 (sem 0.0055) | OOF 0.6023 | folds: 0.5950  0.5960  0.5931  0.6074  0.6226
S5_late3           mean 0.6017 +/- 0.0101 (sem 0.0045) | OOF 0.6015 | folds: 0.5963  0.5963  0.5940  0.6032  0.6187
S_lean_late        mean 0.6016 +/- 0.0124 (sem 0.0056) | OOF 0.6011 | folds: 0.5963  0.5993  0.5870  0.6051  0.6205
S4_late2           mean 0.6016 +/- 0.0121 (sem 0.0054) | OOF 0.6014 | folds: 0.5954  0.5944  0.5912  0.6064  0.6207
S6_secondhalf      mean 0.6000 +/- 0.0134 (sem 0.0060) | OOF 0.5997 | folds: 0.5951  0.5933  0.5869  0.6032  0.6217
S1_uniform         mean 0.5988 +/- 0.0151 (sem 0.0068) | OOF 0.5981 | folds: 0.5946  0.5968  0.5795  0.6016  0.6215
S0_log             mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232
S2_triangular      mean 0.5948 +/- 0.0154 (sem 0.0069) | OOF 0.5941 | folds: 0.5862  0.5908  0.5770  0.6037  0.6163

--- pos-rate -> gain vs log (mechanism check) ---
S0_log           pos-